## M05 - Rankingy acumulados



In [1]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)

ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated


In [2]:
# getOrCreate: si ya hay sesión en este kernel, la reusa (mismo puerto 4040)
spark = get_spark('novashop-clase-m05')
print(spark.version, spark.sparkContext.master)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 16:28:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.5 local[*]


## Cuatro compras, todavía sueltas

In [3]:
from pyspark.sql import Row
from pyspark.sql.functions import col, row_number, sum as fsum
from pyspark.sql.window import Window

# 1 fila = 1 pedido (no una línea de producto)
hist = spark.createDataFrame([
    Row(customer_id="C1", order_id="O1", order_n_ts="2024-01-01", gmv=10.0),
    Row(customer_id="C1", order_id="O2", order_n_ts="2024-02-01", gmv=30.0),
    Row(customer_id="C2", order_id="O3", order_n_ts="2024-01-15", gmv=5.0),
    Row(customer_id="C2", order_id="O4", order_n_ts="2024-03-01", gmv=8.0),
])
print("filas", hist.count())
hist.orderBy("customer_id", "order_n_ts").show()


filas 4
+-----------+--------+----------+----+
|customer_id|order_id|order_n_ts| gmv|
+-----------+--------+----------+----+
|         C1|      O1|2024-01-01|10.0|
|         C1|      O2|2024-02-01|30.0|
|         C2|      O3|2024-01-15| 5.0|
|         C2|      O4|2024-03-01| 8.0|
+-----------+--------+----------+----+



## groupBy. te quedas con el resumen y tirsa el detalle

In [8]:
# una fila por cliente al agrupar por cliente

totales = hist.groupBy("customer_id").agg(fsum("gmv").alias("gmv_total"))
print ("filas después del group by: ", totales.count())
totales.show()



filas después del group by:  2
+-----------+---------+
|customer_id|gmv_total|
+-----------+---------+
|         C1|     40.0|
|         C2|     13.0|
+-----------+---------+



## Ventana: mismas filas, con una columna extra que mira al lado

In [9]:
# Vecindario: mismo cliente, ordenado por fecha
w = Window.partitionBy("customer_id").orderBy("order_n_ts")
ranked = (
    hist.withColumn("order_n", row_number().over(w))   # 1.er, 2.º pedido de ESE cliente
    .withColumn("gmv_running", fsum("gmv").over(w))  # suma desde el 1.er pedido hasta ESTE
)
print("filas (tienen que seguir siendo 4):", ranked.count())
ranked.orderBy("customer_id", "order_n").show()

filas (tienen que seguir siendo 4): 4
+-----------+--------+----------+----+-------+-----------+
|customer_id|order_id|order_n_ts| gmv|order_n|gmv_running|
+-----------+--------+----------+----+-------+-----------+
|         C1|      O1|2024-01-01|10.0|      1|       10.0|
|         C1|      O2|2024-02-01|30.0|      2|       40.0|
|         C2|      O3|2024-01-15| 5.0|      1|        5.0|
|         C2|      O4|2024-03-01| 8.0|      2|       13.0|
+-----------+--------+----------+----+-------+-----------+



## Demo. sin partitionBy el ranking es de toda la empresa


# Sin partitionBy: un solo vecindario (toda la tabla)


In [10]:
w_empresa = Window.orderBy("order_n_ts")
(
    hist.withColumn("order_n", row_number().over(w_empresa))
    .withColumn("gmv_running", fsum("gmv").over(w_empresa))
    .orderBy("order_n")
    .show()
)
# Demuestra: C2 no reinicia en 1. Eso es “olvidaste partitionBy”.

26/09/28 16:43:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 16:43:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 16:43:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+--------+----------+----+-------+-----------+
|customer_id|order_id|order_n_ts| gmv|order_n|gmv_running|
+-----------+--------+----------+----+-------+-----------+
|         C1|      O1|2024-01-01|10.0|      1|       10.0|
|         C2|      O3|2024-01-15| 5.0|      2|       15.0|
|         C1|      O2|2024-02-01|30.0|      3|       45.0|
|         C2|      O4|2024-03-01| 8.0|      4|       53.0|
+-----------+--------+----------+----+-------+-----------+



26/09/28 16:43:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/28 16:43:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## Demo: orderBy al revés acumula desde el pedido más nuevo

In [11]:
# Con partitionBy, orden invertido: el “running” parte del último pedido
w_rev = Window.partitionBy("customer_id").orderBy(col("order_n_ts").desc())
(
    hist.withColumn("order_n", row_number().over(w_rev))
    .withColumn("gmv_running", fsum("gmv").over(w_rev))
    .orderBy("customer_id", "order_n")
    .show()
)
# Demuestra: si gmv_running “no cuadra” con la historia, mira el orderBy de la ventana.


+-----------+--------+----------+----+-------+-----------+
|customer_id|order_id|order_n_ts| gmv|order_n|gmv_running|
+-----------+--------+----------+----+-------+-----------+
|         C1|      O2|2024-02-01|30.0|      1|       30.0|
|         C1|      O1|2024-01-01|10.0|      2|       40.0|
|         C2|      O4|2024-03-01| 8.0|      1|        8.0|
|         C2|      O3|2024-01-15| 5.0|      2|       13.0|
+-----------+--------+----------+----+-------+-----------+

